

Cella 1: Importazione delle Librerie
In questa cella importiamo TensorFlow e i moduli necessari. Notare l'importazione specifica di ResNet50.

In [2]:
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import numpy as np
import matplotlib.pyplot as plt

# Verifica versione e disponibilità GPU (opzionale ma utile)
print(f"TensorFlow Version: {tf.__version__}")

TensorFlow Version: 2.20.0


Cella 2: Configurazione Parametri e Generazione Dati Simulati
Qui definiamo le costanti del progetto (come le dimensioni dell'immagine per ResNet) e creiamo la funzione per generare i dati "dummy" senza dover scaricare dataset reali.

In [3]:
# --- CONFIGURAZIONE PROGETTO ---
IMG_WIDTH, IMG_HEIGHT = 224, 224  # Standard ottimale per ResNet50
IMG_SHAPE = (IMG_WIDTH, IMG_HEIGHT, 3)
NUM_CLASSI = 5                    # Grano, Mais, Soia, Riso, Vuoto
NUM_CAMPIONI = 200                # Numero di immagini simulate
BATCH_SIZE = 32

def genera_dati_agricoli(num_campioni, img_shape, num_classi):
    """
    Simula il dataset di AgriFuture.
    Restituisce tensori di pixel casuali e etichette one-hot encoded.
    """
    print(f"[DATI] Generazione di {num_campioni} campioni simulati (Noise)...")
    
    # X: Immagini simulate (valori random 0-1)
    # Nota: ResNet di solito gradisce preprocessing specifico, ma per la struttura del codice
    # i valori normalizzati 0-1 funzionano tecnicamente per il training flow.
    X = np.random.rand(num_campioni, *img_shape).astype(np.float32)
    
    # y: Etichette (One-Hot Encoding)
    # Crea una matrice di zeri e mette un '1' nella colonna della classe corretta
    indici_classi = np.random.choice(num_classi, num_campioni)
    y = np.eye(num_classi)[indici_classi]
    
    return X, y

# Generazione dei dati
X_train, y_train = genera_dati_agricoli(NUM_CAMPIONI, IMG_SHAPE, NUM_CLASSI)
print(f"Shape Input: {X_train.shape}, Shape Labels: {y_train.shape}")

[DATI] Generazione di 200 campioni simulati (Noise)...
Shape Input: (200, 224, 224, 3), Shape Labels: (200, 5)


Cella 3: Costruzione del Modello (Transfer Learning)
Questa è il cuore dell'esercizio. Carichiamo ResNet50, congeliamo i pesi e aggiungiamo la nuova testa di classificazione per le 5 colture.

In [4]:
def costruisci_modello_resnet(input_shape, num_classi):
    print("[MODELLO] Download/Caricamento pesi ResNet50 (ImageNet)...")
    
    # 1. BASE: ResNet50
    # include_top=False: Tagliamo via i 1000 neuroni originali
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
    
    # 2. FREEZING
    # Congeliamo TUTTI i livelli della base per non distruggere le feature apprese
    base_model.trainable = False
    
    # 3. HEAD (Nuovo Classificatore)
    x = base_model.output
    
    # Pooling: Appiattisce l'output 3D in vettore 1D facendo la media
    x = GlobalAveragePooling2D()(x)
    
    # Dropout: Spegne il 30% dei neuroni per evitare overfitting (regolarizzazione)
    x = Dropout(0.3)(x)
    
    # Output layer: 5 Neuroni (uno per coltura) con Softmax
    outputs = Dense(num_classi, activation='softmax')(x)
    
    model = Model(inputs=base_model.input, outputs=outputs)
    return model

# Istanziazione del modello
model = costruisci_modello_resnet(IMG_SHAPE, NUM_CLASSI)

[MODELLO] Download/Caricamento pesi ResNet50 (ImageNet)...
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 12s 0us/step


Cella 4: Ispezione del Modello
Visualizziamo la struttura. È fondamentale notare la differenza tra Total params (milioni) e Trainable params (pochi, solo quelli della nostra testa).

In [6]:
# Visualizza il sommario
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 23,597,957 (90.02 MB)

 Trainable params: 10,245 (40.02 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

Cella 5: Compilazione e Addestramento
Configuriamo l'ottimizzatore Adam e avviamo il training sulle immagini simulate.

In [7]:
print("[ADDESTRAMENTO] Inizio training su dati simulati...")

# Compilazione
model.compile(optimizer=Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Training
# Validation_split usa il 20% dei dati per testare il modello durante il training
history = model.fit(X_train, y_train, 
                    epochs=5, 
                    batch_size=BATCH_SIZE, 
                    validation_split=0.2)

print("[ADDESTRAMENTO] Completato.")

[ADDESTRAMENTO] Inizio training su dati simulati...
Epoch 1/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 12s 2s/step - accuracy: 0.2375 - loss: 1.7380 - val_accuracy: 0.1250 - val_loss: 1.6514
Epoch 2/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 5s 1s/step - accuracy: 0.2000 - loss: 1.7618 - val_accuracy: 0.1250 - val_loss: 1.6968
Epoch 3/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 5s 1s/step - accuracy: 0.1500 - loss: 1.7890 - val_accuracy: 0.2500 - val_loss: 1.6643
Epoch 4/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 5s 1s/step - accuracy: 0.2500 - loss: 1.6988 - val_accuracy: 0.2500 - val_loss: 1.6215
Epoch 5/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 5s 1s/step - accuracy: 0.2062 - loss: 1.6866 - val_accuracy: 0.2500 - val_loss: 1.5949
[ADDESTRAMENTO] Completato.


Cella 6: (Bonus) Hyperparameter Tuning Semplificato
Qui implementiamo la richiesta facoltativa. Proviamo a creare e addestrare due modelli con Learning Rate diversi per vedere quale converge meglio.

In [8]:
# --- BONUS: TUNING IPERPARAMETRI ---
learning_rates = [0.01, 0.0001]
results = {}

print("\n[BONUS] Avvio ricerca Iperparametri (Grid Search manuale)...")

for lr in learning_rates:
    print(f"\n--- Test con Learning Rate: {lr} ---")
    
    # 1. Ricostruiamo il modello da zero per ogni test
    temp_model = costruisci_modello_resnet(IMG_SHAPE, NUM_CLASSI)
    
    # 2. Compiliamo con il LR corrente
    temp_model.compile(optimizer=Adam(learning_rate=lr),
                       loss='categorical_crossentropy',
                       metrics=['accuracy'])
    
    # 3. Addestriamo brevemente
    h = temp_model.fit(X_train, y_train, epochs=3, batch_size=32, verbose=0, validation_split=0.2)
    
    # Salviamo l'accuratezza finale di validazione
    final_val_acc = h.history['val_accuracy'][-1]
    results[lr] = final_val_acc
    print(f" -> Accuracy Validazione finale con LR {lr}: {final_val_acc:.4f}")

best_lr = max(results, key=results.get)
print(f"\n[RISULTATO BONUS] Il Learning Rate migliore trovato è: {best_lr}")


[BONUS] Avvio ricerca Iperparametri (Grid Search manuale)...

--- Test con Learning Rate: 0.01 ---
[MODELLO] Download/Caricamento pesi ResNet50 (ImageNet)...
 -> Accuracy Validazione finale con LR 0.01: 0.2750

--- Test con Learning Rate: 0.0001 ---
[MODELLO] Download/Caricamento pesi ResNet50 (ImageNet)...
 -> Accuracy Validazione finale con LR 0.0001: 0.1250

[RISULTATO BONUS] Il Learning Rate migliore trovato è: 0.01
